<a href="https://colab.research.google.com/github/Rumas0/Thesis_work_SSL-imbalance/blob/main/SSL_Pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SSL Pretraining**

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

In [10]:
# Setup paths
DRIVE_BACKUP = '/content/drive/MyDrive/thesis_backup_day1/'
os.makedirs(DRIVE_BACKUP, exist_ok=True)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


In [11]:
## Loading Unlabeled Metadata

unlabeled = pd.read_csv('/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/unlabeled_pool_24k.csv')
print(f'Unlabeled Images: {len(unlabeled)}')

unlabeled_sample = unlabeled.sample(n=2000, random_state=42)['image'].tolist()
print(f'Using {len(unlabeled_sample)} for SSL') ## Subset for SSL (Defense purpose n=2000)

Unlabeled Images: 24000
Using 2000 for SSL


# **Generating Synthetic Unlabeled Images**

**500 Images**

In [14]:
unlabeled = pd.read_csv('/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/unlabeled_pool_24k.csv')
unlabeled_sample = unlabeled.sample(n=500, random_state=42)['image'].tolist()
print(f"SSL training on {len(unlabeled_sample)} images")

def create_synthetic_skin_image(path, size=224):
    base = np.random.randint(100, 200, (size, size, 3), dtype=np.uint8)
    x, y = np.meshgrid(np.arange(size), np.arange(size))
    center_x, center_y = np.random.randint(size//3, 2*size//3, 2)
    radius = np.random.randint(30, 80)
    dist = np.sqrt((x-center_x)**2 + (y-center_y)**2)
    mask = dist < radius
    base[mask] = (base[mask] * 0.6).astype(np.uint8)
    noise = np.random.randint(-20, 20, (size, size, 3))
    img = np.clip(base.astype(int) + noise, 0, 255).astype(np.uint8)
    Image.fromarray(img).save(path)

os.makedirs('data/unlabeled', exist_ok=True)
for img_id in tqdm(unlabeled_sample, desc="Creating images"):
    path = f'data/unlabeled/{img_id}.jpg'
    if not os.path.exists(path):
        create_synthetic_skin_image(path)

SSL training on 500 images


Creating images: 100%|██████████| 500/500 [00:00<00:00, 135729.21it/s]


**2000 images**

In [12]:
def create_synthetic_skin_image(path, size=224):
  base = np.random.randint(100, 200, (size, size, 3), dtype=np.uint8)
  x, y = np.meshgrid(np.arange(size), np.arange(size))
  center_x, center_y = np.random.randint(size//3, 2*size//3, 2)
  radius = np.random.randint(30, 80)
  dist = np.sqrt((x-center_x)**2 + (y-center_y)**2)
  mask = dist < radius
  base[mask] = (base[mask] * 0.6).astype(np.uint8)
  noise = np.random.randint(-20, 20, (size, size, 3))
  img = np.clip(base.astype(int) + noise, 0, 255).astype(np.uint8)
  Image.fromarray(img).save(path)

os.makedirs('data/unlabeled', exist_ok=True)

from tqdm import tqdm
for img_id in tqdm(unlabeled_sample):
  path = f'data/unlabeled/{img_id}.jpg'
  if not os.path.exists(path):
    create_synthetic_skin_image(path)

print(f'Created {len(os.listdir('data/unlabeled'))}unlabeled images')

100%|██████████| 2000/2000 [00:00<00:00, 186053.81it/s]

Created 2000unlabeled images


**SimCLR Implementation**

In [18]:
class SimCLRTransform:
    def __init__(self, size=224):
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    def __call__(self, x):
        return self.transform(x), self.transform(x) ##generating 2 views


## Creating Dataset

class UnlabeledDataset(Dataset):
    def __init__(self, image_ids, transform, image_dir='data/unlabeled'):
        self.image_ids = image_ids
        self.transform = transform
        self.image_dir = image_dir
    def __len__(self):
        return len(self.image_ids)
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img = Image.open(f'{self.image_dir}/{img_id}.jpg').convert('RGB')
        return self.transform(img)


## SimCLR Model

class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 112x112
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 56x56
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 28x28
            nn.AdaptiveAvgPool2d(1),  # 128x1x1
            nn.Flatten()  # 128
        )
    def forward(self, x):
        return self.features(x)  # Now returns [B, 128]

class SimCLR(nn.Module):
  def __init__(self, base_encoder, projection_dim=64):
    super().__init__()
    self.encoder = base_encoder
    self.projector = nn.Sequential(
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, projection_dim)
    )

  def forward(self, x):
    h = self.encoder(x)
    z = F.normalize(self.projector(h), dim=1)
    return h, z


## NT-Xent Loss (Contrastive)

def nt_xent_loss(z_i, z_j, temperature=0.5):
    batch_size = z_i.shape[0]
    z = torch.cat([z_i, z_j], dim=0)
    sim_matrix = torch.mm(z, z.t()) / temperature
    mask = torch.eye(2 * batch_size, device=z.device).bool()
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)


 ## Positive Pairs: (i, i+B), (i+B, i)

    pos_sim = torch.cat([torch.diag(sim_matrix, batch_size),
                         torch.diag(sim_matrix, -batch_size)])
    loss = -torch.log(pos_sim / sim_matrix.exp().sum(dim=1))
    return loss.mean()

**Initialize Model**

In [19]:
encoder = SimpleEncoder()  # <-- THIS WAS MISSING IN YOUR CODE
simclr_model = SimCLR(encoder).to(device)

simclr_transform = SimCLRTransform()
unlabeled_dataset = UnlabeledDataset(unlabeled_sample, simclr_transform)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True, drop_last=True)

optimizer = torch.optim.Adam(simclr_model.parameters(), lr=0.001)

#**Training the Loop**

In [20]:
## TRAINING

print('\nTraining SimCLR on unlabeled data...')
epochs = 10

for epoch in range(epochs):
    total_loss = 0
    for view1, view2 in unlabeled_loader:
        view1, view2 = view1.to(device), view2.to(device)

        _, z1 = simclr_model(view1)
        _, z2 = simclr_model(view2)

        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(unlabeled_loader)
    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')

## Saving the pretrained encoder

local_path = 'ssl_encoder_pretrained.pth'
torch.save(encoder.state_dict(), local_path)  # <-- encoder exists now
print(f"\n✓ Saved locally: {local_path}")

drive_path = DRIVE_BACKUP + 'ssl_encoder_pretrained.pth'
torch.save(encoder.state_dict(), drive_path)
print(f"✓ Saved to Drive: {drive_path}")

print(f"\nLocal exists: {os.path.exists(local_path)}")
print(f"Drive exists: {os.path.exists(drive_path)}")

test_load = torch.load(local_path)
print(f"✓ Weights load successfully. Keys: {list(test_load.keys())[:3]}...")


Training SimCLR on unlabeled data...
Epoch 2/10, Loss: 6.0578
Epoch 4/10, Loss: 6.0113
Epoch 6/10, Loss: 5.9876
Epoch 8/10, Loss: 5.9762
Epoch 10/10, Loss: 5.9634

✓ Saved locally: ssl_encoder_pretrained.pth
✓ Saved to Drive: /content/drive/MyDrive/thesis_backup_day1/ssl_encoder_pretrained.pth

Local exists: True
Drive exists: True
✓ Weights load successfully. Keys: ['features.0.weight', 'features.0.bias', 'features.3.weight']...
